In [2]:

from pathlib import Path

# Locate the existing project from the current notebook directory
current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data").exists()
        and (p / "Notebooks 2").exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. "
        "Run this notebook inside the project or a subfolder."
    )

print("Project root:", PROJECT_ROOT)

# Search likely locations without assuming a fixed dataset layout
candidates = [
    PROJECT_ROOT / "Fraiwan",
    PROJECT_ROOT / "data" / "Fraiwan",
    PROJECT_ROOT / "data" / "raw" / "Fraiwan",
    PROJECT_ROOT / "data" / "raw" / "FRAIWAN",
]

fraiwan_paths = [p for p in candidates if p.exists()]

print("\nFraiwan folders found:")
if fraiwan_paths:
    for p in fraiwan_paths:
        print(" -", p)
else:
    print("No standard location found.")

# Also list immediate project folders that may contain the dataset
print("\nProject folders:")
for p in sorted(PROJECT_ROOT.iterdir()):
    if p.is_dir():
        print(" -", p.name)


Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification

Fraiwan folders found:
 - /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/Fraiwan
 - /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/FRAIWAN

Project folders:
 - .git
 - .pytest_cache
 - .ruff_cache
 - .venv
 - .vscode
 - Notebooks 2
 - app
 - data
 - docs
 - models
 - notebooks
 - reports
 - scripts
 - tests


In [3]:

# Cell 2 — Read-only Fraiwan directory audit

from collections import Counter
from pathlib import Path

raw_dir = PROJECT_ROOT / "data" / "raw"

print("========== RAW DATASET FOLDERS ==========")

for name in ["Fraiwan", "FRAIWAN"]:
    path = raw_dir / name
    print(f"\n{name}: exists={path.exists()}")

    if path.exists():
        print("Resolved path:", path.resolve())
        print("Same resolved location as other spelling:",
              any(
                  other.exists() and other.resolve() == path.resolve()
                  for other in [
                      raw_dir / "FRAIWAN",
                      raw_dir / "Fraiwan",
                  ]
                  if other != path
              ))

# Use the existing folder path; do not assume a particular internal layout.
fraiwan_root = raw_dir / "Fraiwan"

if not fraiwan_root.exists():
    fraiwan_root = raw_dir / "FRAIWAN"

if not fraiwan_root.exists():
    raise FileNotFoundError("Fraiwan dataset folder not found.")

print("\n========== DATASET CONTENTS ==========")
print("Dataset root:", fraiwan_root.resolve())

all_files = [
    p for p in fraiwan_root.rglob("*")
    if p.is_file()
]

print("Total files:", len(all_files))

extension_counts = Counter(
    p.suffix.lower() if p.suffix else "[no extension]"
    for p in all_files
)

print("\nFile counts by extension:")
for ext, count in extension_counts.most_common():
    print(f"{ext}: {count}")

print("\nTop-level contents:")
for p in sorted(fraiwan_root.iterdir()):
    if p.is_dir():
        child_count = sum(1 for x in p.rglob("*") if x.is_file())
        print(f"[DIR]  {p.name}/  ({child_count} files)")
    else:
        print(f"[FILE] {p.name}  ({p.stat().st_size:,} bytes)")

print("\nSample file paths (up to 40):")
for p in sorted(all_files)[:40]:
    print(p.relative_to(fraiwan_root))


========== RAW DATASET FOLDERS ==========

Fraiwan: exists=True
Resolved path: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/Fraiwan
Same resolved location as other spelling: False

FRAIWAN: exists=True
Resolved path: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/FRAIWAN
Same resolved location as other spelling: False

========== DATASET CONTENTS ==========
Dataset root: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/Fraiwan
Total files: 339

File counts by extension:
.wav: 336
.xlsx: 1
[no extension]: 1
.zip: 1

Top-level contents:
[FILE] .DS_Store  (8,196 bytes)
[DIR]  Audio Files/  (336 files)
[FILE] Data annotation (1).xlsx  (12,785 bytes)
[FILE] Stethoscope Files.zip  (27,755,197 bytes)

Sample file paths (up to 40):
.DS_Store
Audio Files/BP100_N,N,P R M,70,F.wav
Audio Files/BP101_Asthma,E W,P L M,12,F.wav
Audio Files/BP102_N,N,P L L,41,M.wav
Audio Files/BP103_N,N,P R U,81,F.wav
Audio Files/BP104_Asthma,E W,P L U,45,F.wav
Audio Files/BP105_

In [4]:

# Cell 3 — Fraiwan metadata and filename audit
# Read-only: no audio files or dataset artifacts are modified.

import re
import pandas as pd
from pathlib import Path
from collections import Counter

audio_dir = fraiwan_root / "Audio Files"

xlsx_files = list(fraiwan_root.glob("*.xlsx"))
if not xlsx_files:
    raise FileNotFoundError("No annotation spreadsheet found.")

annotation_path = xlsx_files[0]

print("========== ANNOTATION SPREADSHEET ==========")
print("File:", annotation_path.name)

excel_file = pd.ExcelFile(annotation_path)
print("Sheets:", excel_file.sheet_names)

for sheet in excel_file.sheet_names:
    sheet_df = pd.read_excel(annotation_path, sheet_name=sheet)
    print(f"\n--- Sheet: {sheet} ---")
    print("Shape:", sheet_df.shape)
    print("Columns:", list(sheet_df.columns))
    display(sheet_df.head(8))

print("\n========== WAV FILENAME AUDIT ==========")

wav_paths = sorted(audio_dir.glob("*.wav"))
print("WAV files:", len(wav_paths))

# Filename convention appears to be:
# BP<number>_<diagnosis>,<sound type>,<chest location>,<age>,<sex>.wav
records = []

for path in wav_paths:
    stem = path.stem
    match = re.match(r"^(BP\d+)_([^,]+),(.*)$", stem, flags=re.IGNORECASE)

    if match:
        patient_id = match.group(1).upper()
        remaining = match.group(3).split(",")

        records.append({
            "filename": path.name,
            "patient_id": patient_id,
            "diagnosis_from_filename": match.group(2).strip(),
            "sound_annotation": remaining[0].strip() if len(remaining) > 0 else None,
            "chest_location_raw": remaining[1].strip() if len(remaining) > 1 else None,
            "age_raw": remaining[2].strip() if len(remaining) > 2 else None,
            "sex_raw": remaining[3].strip() if len(remaining) > 3 else None,
            "parse_parts": len(remaining),
        })
    else:
        records.append({
            "filename": path.name,
            "patient_id": None,
            "diagnosis_from_filename": None,
            "sound_annotation": None,
            "chest_location_raw": None,
            "age_raw": None,
            "sex_raw": None,
            "parse_parts": None,
        })

filename_df = pd.DataFrame(records)

print("Parsed filename rows:", len(filename_df))
print("Rows with unparsed patient ID:",
      int(filename_df["patient_id"].isna().sum()))

print("\nUnique parsed patient IDs:",
      filename_df["patient_id"].nunique())

print("\nFiles per patient ID — distribution:")
display(
    filename_df.groupby("patient_id").size()
    .value_counts()
    .sort_index()
    .rename_axis("files_per_patient")
    .to_frame("number_of_patients")
)

print("\nDiagnosis labels from filenames:")
display(
    filename_df["diagnosis_from_filename"]
    .value_counts(dropna=False)
    .to_frame("recordings")
)

print("\nSound annotation labels from filenames:")
display(
    filename_df["sound_annotation"]
    .value_counts(dropna=False)
    .to_frame("recordings")
)

print("\nSample parsed rows:")
display(filename_df.head(15))

print("\nNote: filename-derived counts are preliminary. "
      "The spreadsheet is the source to cross-check before modelling.")


========== ANNOTATION SPREADSHEET ==========
File: Data annotation (1).xlsx
Sheets: ['Sheet1']

--- Sheet: Sheet1 ---
Shape: (154, 10)
Columns: ['Age', 'Gender', 'Location', 'Sound type', 'Diagnosis', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Posterior: P Lower: L Left: L Right R, UPPER: U, ANTERIOR: A, MIDDLE: M']


,Age,Gender,Location,Sound type,Diagnosis,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,"Posterior: P Lower: L Left: L Right R, UPPER: U, ANTERIOR: A, MIDDLE: M"
0,70.0,M,P L L,I E W,Asthma,NaN,NaN,NaN,NaN,"Insipratory: I, Experiatory: E, Wheezes: W, Cr..."
1,52.0,F,P L L,E W,Asthma,NaN,NaN,NaN,NaN,Crep:crepitations
2,50.0,F,P L L,I E W,Asthma,NaN,NaN,NaN,NaN,NaN
3,72.0,F,P R L,I C,Heart Failure + Lung Fibrosis,NaN,NaN,NaN,NaN,NaN
4,71.0,M,P R L,I C E W,Heart Failure + COPD,NaN,NaN,NaN,NaN,NaN
5,81.0,M,P R L,I C B,Plueral Effusion,NaN,NaN,NaN,NaN,NaN
6,65.0,M,P R L,I C E W,Heart Failure + COPD,NaN,NaN,NaN,NaN,NaN
7,70.0,M,P L L,I C B,Plueral Effusion,NaN,NaN,NaN,NaN,NaN



========== WAV FILENAME AUDIT ==========
WAV files: 336
Parsed filename rows: 336
Rows with unparsed patient ID: 224

Unique parsed patient IDs: 112

Files per patient ID — distribution:


,number_of_patients
files_per_patient,
1,112



Diagnosis labels from filenames:


...


Sound annotation labels from filenames:


,recordings
sound_annotation,
NaN,224
E W,39
N,35
Crep,23
C,7
I E W,2
I C E W,2
I C B,2
I C,1



Sample parsed rows:


,filename,patient_id,diagnosis_from_filename,sound_annotation,chest_location_raw,age_raw,sex_raw,parse_parts
0,"BP100_N,N,P R M,70,F.wav",BP100,N,N,P R M,70,F,4.0
1,"BP101_Asthma,E W,P L M,12,F.wav",BP101,Asthma,E W,P L M,12,F,4.0
2,"BP102_N,N,P L L,41,M.wav",BP102,N,N,P L L,41,M,4.0
3,"BP103_N,N,P R U,81,F.wav",BP103,N,N,P R U,81,F,4.0
4,"BP104_Asthma,E W,P L U,45,F.wav",BP104,Asthma,E W,P L U,45,F,4.0
5,"BP105_Lung Fibrosis,Crep,A U R,44,M.wav",BP105,Lung Fibrosis,Crep,A U R,44,M,4.0
6,"BP106_Asthma,E W,P L U,45,F.wav",BP106,Asthma,E W,P L U,45,F,4.0
7,"BP107_Asthma,E W,P L U,59,F.wav",BP107,Asthma,E W,P L U,59,F,4.0
8,"BP108_COPD,E W,P R L ,63,M.wav",BP108,COPD,E W,P R L,63,M,4.0
9,"BP109_N,N,P L M,26,M.wav",BP109,N,N,P L M,26,M,4.0



Note: filename-derived counts are preliminary. The spreadsheet is the source to cross-check before modelling.


In [5]:

# Cell 4 — Inspect filenames missed by the first parser
# Read-only investigation.

import re
import pandas as pd

wav_paths = sorted(audio_dir.glob("*.wav"))

unparsed_paths = [
    path for path in wav_paths
    if not re.match(
        r"^BP\d+_[^,]+,",
        path.stem,
        flags=re.IGNORECASE
    )
]

print("========== UNPARSED FILENAME AUDIT ==========")
print("Unparsed WAV files:", len(unparsed_paths))

print("\nFirst 60 unparsed filenames:")
for path in unparsed_paths[:60]:
    print(path.name)

# Inspect leading filename patterns without assuming their meaning.
prefix_counts = {}
for path in unparsed_paths:
    prefix = re.match(r"^[^_,]+", path.stem)
    prefix = prefix.group(0) if prefix else "[unknown]"
    prefix_counts[prefix] = prefix_counts.get(prefix, 0) + 1

print("\nLeading filename token counts:")
display(
    pd.DataFrame(
        sorted(
            prefix_counts.items(),
            key=lambda item: item[1],
            reverse=True
        ),
        columns=["leading_token", "file_count"]
    )
)

print("\nExamples with commas:")
for path in unparsed_paths:
    if "," in path.stem:
        print(path.name)
        if sum("," in p.stem for p in unparsed_paths[:unparsed_paths.index(path)+1]) >= 15:
            break


========== UNPARSED FILENAME AUDIT ==========
Unparsed WAV files: 224

First 60 unparsed filenames:
DP100_N,N,P R M,70,F.wav
DP101_Asthma,E W,P L M,12,F.wav
DP102_N,N,P L L,41,M.wav
DP103_N,N,P R U,81,F.wav
DP104_Asthma,E W,P L U,45,F.wav
DP105_Lung Fibrosis,Crep,A U R,44,M.wav
DP106_Asthma,E W,P L U,45,F.wav
DP107_Asthma,E W,P L U,59,F.wav
DP108_COPD,E W,P R L ,63,M.wav
DP109_N,N,P L M,26,M.wav
DP10_Asthma,E W,P R U,59,M.wav
DP110_COPD,E W,P L L,62,M.wav
DP111_COPD,E W,P R L ,51,M.wav
DP112_N,N,P L M,30,M.wav
DP11_Heart Failure,C,P L L,53,M.wav
DP12_Asthma,E W,P L L,50,F.wav
DP13_Heart Failure,C,P L L,55,M.wav
DP14_Heart Failure,C,A R M,54,M.wav
DP15_Asthma,E W,P R U,49,F.wav
DP16_heart failure,C,P R U,56,M.wav
DP17_COPD,E W,A R M,57,M.wav
DP18_pneumonia,C,P R U,57,M.wav
DP19_heart failure,C,P R U,70,F.wav
DP1_Asthma,I E W,P L L,70,M.wav
DP20_Asthma and lung fibrosis,C,A R M,90,M.wav
DP21_BRON,Crep,P R L ,20,M.wav
DP22_BRON,Crep,P R U,24,M.wav
DP23_Lung Fibrosis,Crep,P R L ,50,M.wav
D

,leading_token,file_count
0,DP100,1
1,DP101,1
2,DP102,1
3,DP103,1
4,DP104,1
...,...,...
219,EP96,1
220,EP97,1
221,EP98,1
222,EP99,1



Examples with commas:
DP100_N,N,P R M,70,F.wav
DP101_Asthma,E W,P L M,12,F.wav
DP102_N,N,P L L,41,M.wav
DP103_N,N,P R U,81,F.wav
DP104_Asthma,E W,P L U,45,F.wav
DP105_Lung Fibrosis,Crep,A U R,44,M.wav
DP106_Asthma,E W,P L U,45,F.wav
DP107_Asthma,E W,P L U,59,F.wav
DP108_COPD,E W,P R L ,63,M.wav
DP109_N,N,P L M,26,M.wav
DP10_Asthma,E W,P R U,59,M.wav
DP110_COPD,E W,P L L,62,M.wav
DP111_COPD,E W,P R L ,51,M.wav
DP112_N,N,P L M,30,M.wav
DP11_Heart Failure,C,P L L,53,M.wav


In [6]:

# Cell 5 — Robust Fraiwan filename parsing
# Read-only: this cell does not modify files.

import re
import pandas as pd
from pathlib import Path

wav_paths = sorted(audio_dir.glob("*.wav"))
records = []

for path in wav_paths:
    # Accept patient identifiers such as BP100, DP100, EP96, etc.
    match = re.match(
        r"^([A-Za-z]+\d+)_([^,]+),(.*)$",
        path.stem
    )

    if not match:
        records.append({
            "filename": path.name,
            "patient_id": None,
            "diagnosis_raw": None,
            "sound_type": None,
            "location": None,
            "age": None,
            "sex": None,
            "parse_success": False,
        })
        continue

    patient_id = match.group(1).upper()
    diagnosis_raw = match.group(2).strip()
    parts = [x.strip() for x in match.group(3).split(",")]

    records.append({
        "filename": path.name,
        "patient_id": patient_id,
        "diagnosis_raw": diagnosis_raw,
        "sound_type": parts[0] if len(parts) > 0 else None,
        "location": parts[1] if len(parts) > 1 else None,
        "age": parts[2] if len(parts) > 2 else None,
        "sex": parts[3] if len(parts) > 3 else None,
        "parse_success": len(parts) >= 4,
    })

fraiwan_manifest = pd.DataFrame(records)

print("========== PARSER VALIDATION ==========")
print("Total WAV files:", len(fraiwan_manifest))
print("Successfully parsed:", int(fraiwan_manifest["parse_success"].sum()))
print("Unparsed rows:", int((~fraiwan_manifest["parse_success"]).sum()))
print("Unique patient IDs:", fraiwan_manifest["patient_id"].nunique())

print("\n========== PATIENT ID PREFIXES ==========")
fraiwan_manifest["id_prefix"] = (
    fraiwan_manifest["patient_id"]
    .str.extract(r"^([A-Za-z]+)", expand=False)
)

display(
    fraiwan_manifest.groupby("id_prefix", dropna=False)
    .agg(
        recordings=("filename", "size"),
        unique_patients=("patient_id", "nunique"),
    )
)

print("\n========== RAW DIAGNOSIS DISTRIBUTION ==========")
display(
    fraiwan_manifest["diagnosis_raw"]
    .value_counts(dropna=False)
    .to_frame("recordings")
)

print("\n========== PATIENT LABEL CONSISTENCY ==========")
patient_label_counts = (
    fraiwan_manifest.dropna(subset=["patient_id"])
    .groupby("patient_id")["diagnosis_raw"]
    .nunique()
)

print(
    "Patients with multiple filename diagnoses:",
    int((patient_label_counts > 1).sum())
)

print("\n========== DUPLICATE FILENAME CHECK ==========")
print(
    "Duplicate filenames:",
    int(fraiwan_manifest["filename"].duplicated().sum())
)

print("\nSample records:")
display(fraiwan_manifest.head(12))


========== PARSER VALIDATION ==========
Total WAV files: 336
Successfully parsed: 336
Unparsed rows: 0
Unique patient IDs: 336

========== PATIENT ID PREFIXES ==========


,recordings,unique_patients
id_prefix,,
BP,112,112
DP,112,112
EP,112,112



========== RAW DIAGNOSIS DISTRIBUTION ==========


,recordings
diagnosis_raw,
N,105
Asthma,51
heart failure,45
asthma,45
COPD,24
pneumonia,15
Lung Fibrosis,12
Heart Failure,9
BRON,9



========== PATIENT LABEL CONSISTENCY ==========
Patients with multiple filename diagnoses: 0

========== DUPLICATE FILENAME CHECK ==========
Duplicate filenames: 0

Sample records:


,filename,patient_id,diagnosis_raw,sound_type,location,age,sex,parse_success,id_prefix
0,"BP100_N,N,P R M,70,F.wav",BP100,N,N,P R M,70,F,True,BP
1,"BP101_Asthma,E W,P L M,12,F.wav",BP101,Asthma,E W,P L M,12,F,True,BP
2,"BP102_N,N,P L L,41,M.wav",BP102,N,N,P L L,41,M,True,BP
3,"BP103_N,N,P R U,81,F.wav",BP103,N,N,P R U,81,F,True,BP
4,"BP104_Asthma,E W,P L U,45,F.wav",BP104,Asthma,E W,P L U,45,F,True,BP
5,"BP105_Lung Fibrosis,Crep,A U R,44,M.wav",BP105,Lung Fibrosis,Crep,A U R,44,M,True,BP
6,"BP106_Asthma,E W,P L U,45,F.wav",BP106,Asthma,E W,P L U,45,F,True,BP
7,"BP107_Asthma,E W,P L U,59,F.wav",BP107,Asthma,E W,P L U,59,F,True,BP
8,"BP108_COPD,E W,P R L ,63,M.wav",BP108,COPD,E W,P R L,63,M,True,BP
9,"BP109_N,N,P L M,26,M.wav",BP109,N,N,P L M,26,M,True,BP


In [7]:

# Cell 6 — Check whether BP, DP and EP are matched filter versions
# Read-only analysis; no files are modified.

import pandas as pd

df = fraiwan_manifest.copy()

df["id_prefix"] = df["patient_id"].str.extract(
    r"^([A-Za-z]+)", expand=False
)
df["numeric_id"] = pd.to_numeric(
    df["patient_id"].str.extract(r"(\d+)$", expand=False),
    errors="coerce"
)

print("========== PREFIX / NUMERIC-ID COVERAGE ==========")

coverage = pd.crosstab(
    df["numeric_id"],
    df["id_prefix"]
)

print("Numeric IDs:", coverage.shape[0])
print("Prefix columns:", list(coverage.columns))
display(coverage.head(15))

expected_prefixes = {"BP", "DP", "EP"}

print("\nIDs with all three prefixes:",
      int((coverage.reindex(columns=sorted(expected_prefixes), fill_value=0) == 1)
          .all(axis=1).sum()))

print("IDs missing one or more prefixes:",
      int((coverage.reindex(columns=sorted(expected_prefixes), fill_value=0) != 1)
          .any(axis=1).sum()))

# Compare metadata across prefixes for each numeric suffix.
metadata_cols = [
    "diagnosis_raw",
    "sound_type",
    "age",
    "sex",
]

print("\n========== METADATA AGREEMENT BY NUMERIC ID ==========")

for col in metadata_cols:
    counts = (
        df.dropna(subset=["numeric_id"])
        .groupby("numeric_id")[col]
        .nunique(dropna=False)
    )
    print(f"IDs with conflicting {col}: {int((counts > 1).sum())}")

# Show the same numeric IDs across prefixes.
print("\n========== EXAMPLE MATCHED GROUPS ==========")
display(
    df[df["numeric_id"].isin([1, 2, 10, 100])]
    .sort_values(["numeric_id", "id_prefix"])[
        ["filename", "patient_id", "numeric_id", "id_prefix"] + metadata_cols
    ]
)

print(
    "\nImportant: matching numeric IDs and metadata support a possible link, "
    "but do not alone prove the prefix meaning. We will verify against the "
    "annotation sheet and dataset documentation."
)


========== PREFIX / NUMERIC-ID COVERAGE ==========
Numeric IDs: 112
Prefix columns: ['BP', 'DP', 'EP']


id_prefix,BP,DP,EP
numeric_id,,,
1,1,1,1
2,1,1,1
3,1,1,1
4,1,1,1
5,1,1,1
6,1,1,1
7,1,1,1
8,1,1,1
9,1,1,1



IDs with all three prefixes: 112
IDs missing one or more prefixes: 0

========== METADATA AGREEMENT BY NUMERIC ID ==========
IDs with conflicting diagnosis_raw: 0
IDs with conflicting sound_type: 0
IDs with conflicting age: 0
IDs with conflicting sex: 0

========== EXAMPLE MATCHED GROUPS ==========


,filename,patient_id,numeric_id,id_prefix,diagnosis_raw,sound_type,age,sex
23,"BP1_Asthma,I E W,P L L,70,M.wav",BP1,1,BP,Asthma,I E W,70,M
135,"DP1_Asthma,I E W,P L L,70,M.wav",DP1,1,DP,Asthma,I E W,70,M
247,"EP1_Asthma,I E W,P L L,70,M.wav",EP1,1,EP,Asthma,I E W,70,M
34,"BP2_Asthma,E W,P L L R,52,F.wav",BP2,2,BP,Asthma,E W,52,F
146,"DP2_Asthma,E W,P L L R,52,F.wav",DP2,2,DP,Asthma,E W,52,F
258,"EP2_Asthma,E W,P L L R,52,F.wav",EP2,2,EP,Asthma,E W,52,F
10,"BP10_Asthma,E W,P R U,59,M.wav",BP10,10,BP,Asthma,E W,59,M
122,"DP10_Asthma,E W,P R U,59,M.wav",DP10,10,DP,Asthma,E W,59,M
234,"EP10_Asthma,E W,P R U,59,M.wav",EP10,10,EP,Asthma,E W,59,M
0,"BP100_N,N,P R M,70,F.wav",BP100,100,BP,N,N,70,F



Important: matching numeric IDs and metadata support a possible link, but do not alone prove the prefix meaning. We will verify against the annotation sheet and dataset documentation.


In [8]:

# Cell 7 — Inspect annotation rows and compare with audio metadata

import pandas as pd
from pathlib import Path

excel_path = fraiwan_root / "Data annotation (1).xlsx"
annotations = pd.read_excel(excel_path, sheet_name="Sheet1")

print("========== ANNOTATION SHEET ==========")
print("Shape:", annotations.shape)
print("\nColumns:")
for col in annotations.columns:
    print(f"- {col!r}")

print("\n========== NON-EMPTY COUNTS ==========")
display(
    pd.DataFrame({
        "non_empty": annotations.notna().sum(),
        "unique_values": annotations.nunique(dropna=True)
    }).sort_values("non_empty", ascending=False)
)

print("\n========== DIAGNOSIS VALUES IN ANNOTATION SHEET ==========")
diagnosis_col = next(
    (c for c in annotations.columns
     if str(c).strip().lower() == "diagnosis"),
    None
)

if diagnosis_col is not None:
    print(annotations[diagnosis_col].value_counts(dropna=False).to_string())
else:
    print("No column named 'Diagnosis' was found.")

print("\n========== FIRST 20 ANNOTATION ROWS ==========")
display(annotations.head(20))

print("\n========== AUDIO DIAGNOSIS COUNTS (UNIQUE SUBJECTS) ==========")
# Each numeric ID represents one candidate underlying subject.
audio_subjects = (
    fraiwan_manifest
    .assign(
        numeric_id=pd.to_numeric(
            fraiwan_manifest["patient_id"]
            .str.extract(r"(\d+)$", expand=False),
            errors="coerce"
        )
    )
    .sort_values("filename")
    .drop_duplicates("numeric_id")
)

display(
    audio_subjects["diagnosis_raw"]
    .value_counts(dropna=False)
    .rename_axis("diagnosis_raw")
    .to_frame("subjects")
)

print("\nNote: the spreadsheet may annotate recording locations or sound events "
      "rather than provide one row for every WAV file. We will determine its "
      "relationship to the filenames before joining anything.")


========== ANNOTATION SHEET ==========
Shape: (154, 10)

Columns:
- 'Age'
- 'Gender'
- 'Location'
- 'Sound type'
- 'Diagnosis'
- 'Unnamed: 5'
- 'Unnamed: 6'
- 'Unnamed: 7'
- 'Unnamed: 8'
- 'Posterior: P Lower: L Left: L Right R, UPPER: U, ANTERIOR: A, MIDDLE: M'

========== NON-EMPTY COUNTS ==========


,non_empty,unique_values
Age,112,52
Gender,112,2
Location,112,11
Sound type,112,9
Diagnosis,112,14
Unnamed: 5,2,1
Unnamed: 6,2,1
"Posterior: P Lower: L Left: L Right R, UPPER: U, ANTERIOR: A, MIDDLE: M",2,2
Unnamed: 7,0,0
Unnamed: 8,0,0



========== DIAGNOSIS VALUES IN ANNOTATION SHEET ==========
Diagnosis
NaN                               42
N                                 35
Asthma                            17
heart failure                     15
asthma                            15
COPD                               8
pneumonia                          5
Lung Fibrosis                      4
Heart Failure                      3
BRON                               3
Heart Failure + COPD               2
Plueral Effusion                   2
Heart Failure + Lung Fibrosis      1
Asthma and lung fibrosis           1
copd                               1

========== FIRST 20 ANNOTATION ROWS ==========


,Age,Gender,Location,Sound type,Diagnosis,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,"Posterior: P Lower: L Left: L Right R, UPPER: U, ANTERIOR: A, MIDDLE: M"
0,70.0,M,P L L,I E W,Asthma,NaN,NaN,NaN,NaN,"Insipratory: I, Experiatory: E, Wheezes: W, Cr..."
1,52.0,F,P L L,E W,Asthma,NaN,NaN,NaN,NaN,Crep:crepitations
2,50.0,F,P L L,I E W,Asthma,NaN,NaN,NaN,NaN,NaN
3,72.0,F,P R L,I C,Heart Failure + Lung Fibrosis,NaN,NaN,NaN,NaN,NaN
4,71.0,M,P R L,I C E W,Heart Failure + COPD,NaN,NaN,NaN,NaN,NaN
5,81.0,M,P R L,I C B,Plueral Effusion,NaN,NaN,NaN,NaN,NaN
6,65.0,M,P R L,I C E W,Heart Failure + COPD,NaN,NaN,NaN,NaN,NaN
7,70.0,M,P L L,I C B,Plueral Effusion,NaN,NaN,NaN,NaN,NaN
8,59.0,M,P R L,E W,Asthma,NaN,NaN,NaN,NaN,NaN
9,59.0,M,P R U,E W,Asthma,NaN,NaN,NaN,NaN,NaN



========== AUDIO DIAGNOSIS COUNTS (UNIQUE SUBJECTS) ==========


,subjects
diagnosis_raw,
N,35
Asthma,17
heart failure,15
asthma,15
COPD,8
pneumonia,5
Lung Fibrosis,4
Heart Failure,3
BRON,3



Note: the spreadsheet may annotate recording locations or sound events rather than provide one row for every WAV file. We will determine its relationship to the filenames before joining anything.


In [9]:

# Cell 8 — Compare spreadsheet records with unique audio-subject metadata

import pandas as pd
import re

# Build one row per numeric ID from the WAV manifest.
audio = fraiwan_manifest.copy()
audio["numeric_id"] = pd.to_numeric(
    audio["patient_id"].str.extract(r"(\d+)$", expand=False),
    errors="coerce"
)

audio_subjects = (
    audio.sort_values(["numeric_id", "id_prefix"])
    .drop_duplicates("numeric_id")
    .copy()
)

# Normalize text only for comparison; preserve original values in source data.
def norm_text(value):
    if pd.isna(value):
        return None
    return re.sub(r"\s+", " ", str(value).strip()).casefold()

def norm_age(value):
    if pd.isna(value):
        return None
    try:
        return int(float(value))
    except (ValueError, TypeError):
        return norm_text(value)

# Keep populated rows; the spreadsheet has no explicit subject-ID column.
sheet = annotations.copy()
sheet = sheet[sheet["Age"].notna()].reset_index(drop=True)

print("========== RECORD COUNTS ==========")
print("Unique numeric audio IDs:", audio_subjects["numeric_id"].nunique())
print("Spreadsheet rows with age:", len(sheet))
print("Spreadsheet rows with complete core fields:",
      int(sheet[["Age", "Gender", "Location", "Sound type", "Diagnosis"]]
          .notna().all(axis=1).sum()))

print("\n========== UNIQUE-VALUE COUNTS ==========")
for col in ["Age", "Gender", "Location", "Sound type", "Diagnosis"]:
    print(
        f"{col}: spreadsheet={sheet[col].nunique(dropna=True)}, "
        f"audio={audio_subjects[{
            'Age': 'age',
            'Gender': 'sex',
            'Location': 'location',
            'Sound type': 'sound_type',
            'Diagnosis': 'diagnosis_raw'
        }[col]].nunique(dropna=True)}"
    )

# Compare metadata distributions, not row order.
column_map = {
    "Age": "age",
    "Gender": "sex",
    "Location": "location",
    "Sound type": "sound_type",
    "Diagnosis": "diagnosis_raw",
}

print("\n========== DISTRIBUTION AGREEMENT ==========")
for sheet_col, audio_col in column_map.items():
    left = sheet[sheet_col].map(
        norm_age if sheet_col == "Age" else norm_text
    ).value_counts(dropna=False).sort_index(key=lambda x: x.astype(str))

    right = audio_subjects[audio_col].map(
        norm_age if sheet_col == "Age" else norm_text
    ).value_counts(dropna=False).sort_index(key=lambda x: x.astype(str))

    same = left.equals(right)
    print(f"{sheet_col}: distributions identical = {same}")
    if not same:
        comparison = pd.concat(
            [left.rename("spreadsheet"), right.rename("audio")],
            axis=1
        ).fillna(0).astype(int)
        display(comparison[comparison["spreadsheet"] != comparison["audio"]])

print("\nImportant: identical distributions would support consistency, "
      "but would not prove that spreadsheet row i belongs to numeric ID i+1.")


========== RECORD COUNTS ==========
Unique numeric audio IDs: 112
Spreadsheet rows with age: 112
Spreadsheet rows with complete core fields: 112

========== UNIQUE-VALUE COUNTS ==========
Age: spreadsheet=52, audio=52
Gender: spreadsheet=2, audio=2
Location: spreadsheet=11, audio=14
Sound type: spreadsheet=9, audio=9
Diagnosis: spreadsheet=14, audio=14

========== DISTRIBUTION AGREEMENT ==========
Age: distributions identical = True
Gender: distributions identical = True
Location: distributions identical = False


,spreadsheet,audio
a r u,6,5
p l l,19,16
p r l,24,20
a u r,0,1
p,0,1
p l l r,0,2
p l r,0,4


Sound type: distributions identical = True
Diagnosis: distributions identical = True

Important: identical distributions would support consistency, but would not prove that spreadsheet row i belongs to numeric ID i+1.


In [10]:

# Cell 9 — Inspect location differences and compare subject metadata

import pandas as pd
import re

def clean_label(value):
    if pd.isna(value):
        return None
    return re.sub(r"\s+", " ", str(value).strip()).upper()

audio = fraiwan_manifest.copy()
audio["numeric_id"] = pd.to_numeric(
    audio["patient_id"].str.extract(r"(\d+)$", expand=False),
    errors="coerce"
)
audio["location_clean"] = audio["location"].map(clean_label)

# One row per candidate subject, using BP as the representative version.
audio_subjects = (
    audio[audio["id_prefix"] == "BP"]
    .sort_values("numeric_id")
    .drop_duplicates("numeric_id")
    .copy()
)

sheet = annotations[annotations["Age"].notna()].copy().reset_index(drop=True)
sheet["location_clean"] = sheet["Location"].map(clean_label)

print("========== LOCATION VALUES: SPREADSHEET ==========")
display(
    sheet["location_clean"].value_counts(dropna=False)
    .rename_axis("location")
    .to_frame("count")
)

print("\n========== LOCATION VALUES: AUDIO ==========")
display(
    audio_subjects["location_clean"].value_counts(dropna=False)
    .rename_axis("location")
    .to_frame("count")
)

print("\n========== RAW LOCATION STRINGS BY PREFIX ==========")
display(
    audio.groupby(["id_prefix", "location"], dropna=False)
    .size()
    .rename("recordings")
    .reset_index()
    .sort_values(["id_prefix", "location"])
)

print("\n========== AUDIO METADATA AGREEMENT ACROSS BP/DP/EP ==========")
for col in ["age", "sex", "location", "sound_type", "diagnosis_raw"]:
    conflicts = (
        audio.groupby("numeric_id")[col]
        .nunique(dropna=False)
        .gt(1)
    )
    print(f"{col}: conflicting numeric IDs = {int(conflicts.sum())}")

print("\n========== FIRST 15 SUBJECTS: AUDIO METADATA ==========")
display(
    audio_subjects[
        ["numeric_id", "filename", "age", "sex",
         "location", "sound_type", "diagnosis_raw"]
    ].head(15)
)

print("\n========== FIRST 15 POPULATED SPREADSHEET ROWS ==========")
display(
    sheet[
        ["Age", "Gender", "Location", "Sound type", "Diagnosis"]
    ].head(15)
)

print(
    "\nDo not join spreadsheet rows to audio IDs by row order yet. "
    "We are checking whether the metadata provides enough evidence "
    "to establish a safe mapping."
)


========== LOCATION VALUES: SPREADSHEET ==========


,count
location,
P R L,24
P L L,19
P R M,16
P R U,14
P L M,12
P L U,11
A R U,6
A R M,4
A R L,4



========== LOCATION VALUES: AUDIO ==========


,count
location,
P R L,20
P L L,16
P R M,16
P R U,14
P L M,12
P L U,11
A R U,5
P L R,4
A R M,4



========== RAW LOCATION STRINGS BY PREFIX ==========


...


========== AUDIO METADATA AGREEMENT ACROSS BP/DP/EP ==========
age: conflicting numeric IDs = 0
sex: conflicting numeric IDs = 0
location: conflicting numeric IDs = 0
sound_type: conflicting numeric IDs = 0
diagnosis_raw: conflicting numeric IDs = 0

========== FIRST 15 SUBJECTS: AUDIO METADATA ==========


,numeric_id,filename,age,sex,location,sound_type,diagnosis_raw
23,1,"BP1_Asthma,I E W,P L L,70,M.wav",70,M,P L L,I E W,Asthma
34,2,"BP2_Asthma,E W,P L L R,52,F.wav",52,F,P L L R,E W,Asthma
45,3,"BP3_Asthma,I E W,P L L R,50,F.wav",50,F,P L L R,I E W,Asthma
56,4,"BP4_Heart Failure + Lung Fibrosis ,I C,P L R,7...",72,F,P L R,I C,Heart Failure + Lung Fibrosis
67,5,"BP5_Heart Failure + COPD,I C E W,P L R ,71,M.wav",71,M,P L R,I C E W,Heart Failure + COPD
78,6,"BP6_Plueral Effusion,I C B,P L R,81,M.wav",81,M,P L R,I C B,Plueral Effusion
89,7,"BP7_Heart Failure + COPD,I C E W,P L R,65,M.wav",65,M,P L R,I C E W,Heart Failure + COPD
100,8,"BP8_Plueral Effusion,I C B,P L L,70,M.wav",70,M,P L L,I C B,Plueral Effusion
111,9,"BP9_Asthma,E W,P R L ,59,M.wav",59,M,P R L,E W,Asthma
10,10,"BP10_Asthma,E W,P R U,59,M.wav",59,M,P R U,E W,Asthma



========== FIRST 15 POPULATED SPREADSHEET ROWS ==========


,Age,Gender,Location,Sound type,Diagnosis
0,70.0,M,P L L,I E W,Asthma
1,52.0,F,P L L,E W,Asthma
2,50.0,F,P L L,I E W,Asthma
3,72.0,F,P R L,I C,Heart Failure + Lung Fibrosis
4,71.0,M,P R L,I C E W,Heart Failure + COPD
5,81.0,M,P R L,I C B,Plueral Effusion
6,65.0,M,P R L,I C E W,Heart Failure + COPD
7,70.0,M,P L L,I C B,Plueral Effusion
8,59.0,M,P R L,E W,Asthma
9,59.0,M,P R U,E W,Asthma



Do not join spreadsheet rows to audio IDs by row order yet. We are checking whether the metadata provides enough evidence to establish a safe mapping.


In [11]:

# Cell 10 — Verify spreadsheet row order against all 112 BP subject records

import pandas as pd
import re

def norm(value):
    if pd.isna(value):
        return None
    return re.sub(r"\s+", " ", str(value).strip()).casefold()

def norm_age(value):
    if pd.isna(value):
        return None
    try:
        return int(float(value))
    except (ValueError, TypeError):
        return norm(value)

# One BP recording per numeric ID, ordered by numeric ID.
audio = fraiwan_manifest.copy()
audio["numeric_id"] = pd.to_numeric(
    audio["patient_id"].str.extract(r"(\d+)$", expand=False),
    errors="coerce"
)

bp = (
    audio[audio["id_prefix"] == "BP"]
    .sort_values("numeric_id")
    .drop_duplicates("numeric_id")
    .reset_index(drop=True)
)

# The 112 populated spreadsheet rows, preserving their original order.
sheet = (
    annotations[annotations["Age"].notna()]
    .reset_index()
    .rename(columns={"index": "excel_row_index"})
)

print("Audio subjects:", len(bp))
print("Populated annotation rows:", len(sheet))

# This is a diagnostic comparison, not an assumption that row order is valid.
field_pairs = {
    "age": ("Age", "age", norm_age),
    "sex": ("Gender", "sex", norm),
    "sound_type": ("Sound type", "sound_type", norm),
    "diagnosis": ("Diagnosis", "diagnosis_raw", norm),
    "location": ("Location", "location", norm),
}

results = pd.DataFrame({
    "numeric_id": bp["numeric_id"],
    "filename": bp["filename"],
})

for field, (sheet_col, audio_col, normalize) in field_pairs.items():
    sheet_values = sheet[sheet_col].map(normalize).tolist()
    audio_values = bp[audio_col].map(normalize).tolist()

    results[f"{field}_match"] = [
        a == b for a, b in zip(sheet_values, audio_values)
    ]
    results[f"{field}_sheet"] = sheet_values
    results[f"{field}_audio"] = audio_values

print("\n========== MATCH RATES IF ROW ORDER IS USED ==========")
for field in field_pairs:
    col = f"{field}_match"
    matches = int(results[col].sum())
    print(f"{field}: {matches}/{len(results)} ({matches / len(results):.1%})")

core_cols = ["age_match", "sex_match", "sound_type_match", "diagnosis_match"]
results["core_match"] = results[core_cols].all(axis=1)

print("\nAll four core fields match:",
      int(results["core_match"].sum()), "/", len(results))
print("All five fields including location match:",
      int(results[[f"{field}_match" for field in field_pairs]].all(axis=1).sum()),
      "/", len(results))

print("\n========== CORE-FIELD MISMATCHES ==========")
display(
    results.loc[
        ~results["core_match"],
        ["numeric_id", "filename",
         "age_sheet", "age_audio",
         "sex_sheet", "sex_audio",
         "sound_type_sheet", "sound_type_audio",
         "diagnosis_sheet", "diagnosis_audio"]
    ].head(30)
)

print("\n========== LOCATION MISMATCHES ==========")
display(
    results.loc[
        ~results["location_match"],
        ["numeric_id", "filename", "location_sheet", "location_audio"]
    ].head(30)
)

print(
    "\nInterpretation: a perfect core-field match across all rows would "
    "strongly support the row-order mapping, although a documented subject "
    "identifier would still be preferable."
)


Audio subjects: 112
Populated annotation rows: 112

========== MATCH RATES IF ROW ORDER IS USED ==========
age: 112/112 (100.0%)
sex: 112/112 (100.0%)
sound_type: 112/112 (100.0%)
diagnosis: 112/112 (100.0%)
location: 104/112 (92.9%)

All four core fields match: 112 / 112
All five fields including location match: 104 / 112

========== CORE-FIELD MISMATCHES ==========


,numeric_id,filename,age_sheet,age_audio,sex_sheet,sex_audio,sound_type_sheet,sound_type_audio,diagnosis_sheet,diagnosis_audio



========== LOCATION MISMATCHES ==========


,numeric_id,filename,location_sheet,location_audio
1,2,"BP2_Asthma,E W,P L L R,52,F.wav",p l l,p l l r
2,3,"BP3_Asthma,I E W,P L L R,50,F.wav",p l l,p l l r
3,4,"BP4_Heart Failure + Lung Fibrosis ,I C,P L R,7...",p r l,p l r
4,5,"BP5_Heart Failure + COPD,I C E W,P L R ,71,M.wav",p r l,p l r
5,6,"BP6_Plueral Effusion,I C B,P L R,81,M.wav",p r l,p l r
6,7,"BP7_Heart Failure + COPD,I C E W,P L R,65,M.wav",p r l,p l r
25,26,"BP26_Lung Fibrosis,Crep,P,90,F.wav",p l l,p
104,105,"BP105_Lung Fibrosis,Crep,A U R,44,M.wav",a r u,a u r



Interpretation: a perfect core-field match across all rows would strongly support the row-order mapping, although a documented subject identifier would still be preferable.


In [12]:

# Cell 11 — Audit Fraiwan diagnoses against the eight ICBHI classes

import pandas as pd
import re

# The eight classes used in our ICBHI project.
icbhi_classes = {
    "asthma",
    "bronchiectasis",
    "bronchiolitis",
    "copd",
    "healthy",
    "lrti",
    "pneumonia",
    "urti",
}

# Count each original Fraiwan diagnosis at the subject level.
subjects = (
    fraiwan_manifest.assign(
        numeric_id=pd.to_numeric(
            fraiwan_manifest["patient_id"]
            .str.extract(r"(\d+)$", expand=False),
            errors="coerce"
        )
    )
    .sort_values(["numeric_id", "id_prefix"])
    .drop_duplicates("numeric_id")
    .copy()
)

def normalize_diagnosis(value):
    if pd.isna(value):
        return ""
    value = re.sub(r"\s+", " ", str(value).strip()).casefold()
    return value

subjects["diagnosis_normalized"] = subjects["diagnosis_raw"].map(
    normalize_diagnosis
)

diagnosis_counts = (
    subjects.groupby(["diagnosis_raw", "diagnosis_normalized"], dropna=False)
    .size()
    .reset_index(name="subjects")
    .sort_values("subjects", ascending=False)
)

def classify_mapping(diagnosis):
    d = normalize_diagnosis(diagnosis)

    # Exact/obvious label matches only; no speculative equivalences.
    if d in {"asthma", "copd", "pneumonia"}:
        return "Direct label candidate"
    if d in {"n", "normal", "healthy"}:
        return "Potential healthy candidate — verify definition"
    if not d:
        return "Missing diagnosis"
    if "+" in d or " and " in d:
        return "Combined diagnosis — not directly mappable"
    if d == "bron":
        return "Ambiguous abbreviation — verify source definition"
    return "No direct ICBHI class match"

diagnosis_counts["mapping_status"] = diagnosis_counts["diagnosis_raw"].map(
    classify_mapping
)

print("========== DIAGNOSIS MAPPING AUDIT ==========")
display(diagnosis_counts)

print("\n========== SUBJECT COUNTS BY MAPPING STATUS ==========")
display(
    diagnosis_counts.groupby("mapping_status", dropna=False)["subjects"]
    .sum()
    .sort_values(ascending=False)
    .to_frame("subjects")
)

print("\n========== ICBHI CLASSES WITHOUT AN OBVIOUS FRAIWAN MATCH ==========")
obvious = {"asthma", "copd", "pneumonia", "healthy"}
print(sorted(icbhi_classes - obvious))

print(
    "\nThis is a conservative label-compatibility audit, not a final mapping. "
    "In particular, N/normal must be checked against the dataset's definition "
    "before treating it as equivalent to ICBHI Healthy."
)


========== DIAGNOSIS MAPPING AUDIT ==========


,diagnosis_raw,diagnosis_normalized,subjects,mapping_status
8,N,n,35,Potential healthy candidate — verify definition
0,Asthma,asthma,17,Direct label candidate
10,asthma,asthma,15,Direct label candidate
12,heart failure,heart failure,15,No direct ICBHI class match
3,COPD,copd,8,Direct label candidate
13,pneumonia,pneumonia,5,Direct label candidate
7,Lung Fibrosis,lung fibrosis,4,No direct ICBHI class match
2,BRON,bron,3,Ambiguous abbreviation — verify source definition
4,Heart Failure,heart failure,3,No direct ICBHI class match
5,Heart Failure + COPD,heart failure + copd,2,Combined diagnosis — not directly mappable



========== SUBJECT COUNTS BY MAPPING STATUS ==========


,subjects
mapping_status,
Direct label candidate,46
Potential healthy candidate — verify definition,35
No direct ICBHI class match,24
Combined diagnosis — not directly mappable,4
Ambiguous abbreviation — verify source definition,3



========== ICBHI CLASSES WITHOUT AN OBVIOUS FRAIWAN MATCH ==========
['bronchiectasis', 'bronchiolitis', 'lrti', 'urti']

This is a conservative label-compatibility audit, not a final mapping. In particular, N/normal must be checked against the dataset's definition before treating it as equivalent to ICBHI Healthy.


In [13]:

# Cell 12 — Audit audio format, duration, amplitude, and version differences

from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import soundfile as sf

records = []
errors = []

for path in sorted(fraiwan_root.rglob("*.wav")):
    try:
        info = sf.info(str(path))
        signal, sr = sf.read(str(path), dtype="float32", always_2d=True)

        peak = float(np.max(np.abs(signal))) if signal.size else 0.0
        rms = float(np.sqrt(np.mean(signal ** 2))) if signal.size else 0.0

        name = path.name
        prefix_match = __import__("re").match(r"^([A-Za-z]+)(\d+)_", name)

        records.append({
            "filename": name,
            "path": str(path),
            "prefix": prefix_match.group(1) if prefix_match else None,
            "numeric_id": int(prefix_match.group(2)) if prefix_match else None,
            "sample_rate": sr,
            "channels": signal.shape[1],
            "frames": signal.shape[0],
            "duration_seconds": signal.shape[0] / sr if sr else np.nan,
            "subtype": info.subtype,
            "peak_amplitude": peak,
            "rms_amplitude": rms,
            "finite_samples": bool(np.isfinite(signal).all()),
            "silent": bool(rms == 0.0),
            "samples": signal[:, 0].copy() if signal.shape[1] == 1 else None,
        })

    except Exception as exc:
        errors.append({"filename": path.name, "error": repr(exc)})

audio_audit = pd.DataFrame(records)

print("========== FILE READABILITY ==========")
print("WAV files inspected:", len(audio_audit))
print("Read errors:", len(errors))
if errors:
    display(pd.DataFrame(errors).head(20))

print("\n========== SAMPLE RATE COUNTS ==========")
display(audio_audit["sample_rate"].value_counts(dropna=False).to_frame("files"))

print("\n========== CHANNEL COUNTS ==========")
display(audio_audit["channels"].value_counts(dropna=False).to_frame("files"))

print("\n========== DURATION SUMMARY ==========")
display(
    audio_audit["duration_seconds"]
    .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
    .to_frame("seconds")
)

print("\n========== AUDIO QUALITY FLAGS ==========")
print("Non-finite signals:", int((~audio_audit["finite_samples"]).sum()))
print("Silent signals:", int(audio_audit["silent"].sum()))
print("Peak amplitude >= 0.999:", int((audio_audit["peak_amplitude"] >= 0.999).sum()))

print("\n========== FORMAT AND SIGNAL SUMMARY BY PREFIX ==========")
display(
    audio_audit.groupby("prefix").agg(
        files=("filename", "count"),
        sample_rates=("sample_rate", lambda x: sorted(x.dropna().unique().tolist())),
        channel_counts=("channels", lambda x: sorted(x.dropna().unique().tolist())),
        median_duration=("duration_seconds", "median"),
        min_duration=("duration_seconds", "min"),
        max_duration=("duration_seconds", "max"),
        median_rms=("rms_amplitude", "median"),
        median_peak=("peak_amplitude", "median"),
    )
)

print("\n========== BP / DP / EP SIGNAL COMPARISON ==========")
# Compare aligned versions for each numeric ID when their sample counts and
# sample rates match. Correlation is descriptive, not proof of filter identity.
comparison = []

for numeric_id, group in audio_audit.dropna(subset=["numeric_id"]).groupby("numeric_id"):
    by_prefix = {row["prefix"]: row for _, row in group.iterrows()}
    if not {"BP", "DP", "EP"}.issubset(by_prefix):
        continue

    for a, b in [("BP", "DP"), ("BP", "EP"), ("DP", "EP")]:
        ra, rb = by_prefix[a], by_prefix[b]
        comparable = (
            ra["sample_rate"] == rb["sample_rate"]
            and ra["frames"] == rb["frames"]
            and ra["samples"] is not None
            and rb["samples"] is not None
        )

        corr = np.nan
        max_abs_diff = np.nan
        if comparable:
            xa, xb = ra["samples"], rb["samples"]
            if np.std(xa) > 0 and np.std(xb) > 0:
                corr = float(np.corrcoef(xa, xb)[0, 1])
            max_abs_diff = float(np.max(np.abs(xa - xb)))

        comparison.append({
            "numeric_id": int(numeric_id),
            "pair": f"{a}-{b}",
            "comparable_shape": comparable,
            "correlation": corr,
            "max_absolute_difference": max_abs_diff,
        })

comparison_df = pd.DataFrame(comparison)
display(comparison_df.head(15))

print("\nPair counts:")
display(comparison_df["pair"].value_counts().to_frame("pairs"))

print(
    "\nNote: correlation and amplitude differences help identify signal changes, "
    "but the prefixes' filter meanings must be confirmed from dataset documentation."
)



========== FILE READABILITY ==========
WAV files inspected: 336
Read errors: 0

========== SAMPLE RATE COUNTS ==========


,files
sample_rate,
4000,336



========== CHANNEL COUNTS ==========


,files
channels,
1,336



========== DURATION SUMMARY ==========


...


========== BP / DP / EP SIGNAL COMPARISON ==========


,numeric_id,pair,comparable_shape,correlation,max_absolute_difference
0,1,BP-DP,True,0.742706,0.777374
1,1,BP-EP,True,0.822664,0.615692
2,1,DP-EP,True,0.940444,0.346313
3,2,BP-DP,True,0.718432,0.949829
4,2,BP-EP,True,0.767174,0.831543
5,2,DP-EP,True,0.925225,0.731079
6,3,BP-DP,True,0.733483,0.804871
7,3,BP-EP,True,0.798469,0.769409
8,3,DP-EP,True,0.932956,0.236053
9,4,BP-DP,True,0.738873,0.672699



Pair counts:


,pairs
pair,
BP-DP,112
BP-EP,112
DP-EP,112



Note: correlation and amplitude differences help identify signal changes, but the prefixes' filter meanings must be confirmed from dataset documentation.


In [14]:

# Cell 13 — Check clipping/saturation and summarize all paired signal differences

import numpy as np
import pandas as pd
import soundfile as sf
import re

audio_files = sorted(fraiwan_root.rglob("*.wav"))
signal_stats = []
paired_stats = []

signals_by_id = {}

for path in audio_files:
    match = re.match(r"^([A-Za-z]+)(\d+)_", path.name)
    if not match:
        continue

    prefix = match.group(1)
    numeric_id = int(match.group(2))

    signal, sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = signal[:, 0]

    signal_stats.append({
        "numeric_id": numeric_id,
        "prefix": prefix,
        "filename": path.name,
        "sample_rate": sr,
        "samples": len(x),
        "duration": len(x) / sr,
        "peak": float(np.max(np.abs(x))),
        "fraction_abs_ge_0999": float(np.mean(np.abs(x) >= 0.999)),
        "fraction_abs_ge_099": float(np.mean(np.abs(x) >= 0.99)),
        "fraction_at_positive_limit": float(np.mean(x >= 0.999)),
        "fraction_at_negative_limit": float(np.mean(x <= -0.999)),
        "rms": float(np.sqrt(np.mean(x ** 2))),
    })

    signals_by_id.setdefault(numeric_id, {})[prefix] = {
        "signal": x,
        "sample_rate": sr,
        "duration": len(x) / sr,
    }

stats_df = pd.DataFrame(signal_stats)

print("========== SAMPLE SATURATION SUMMARY ==========")
display(
    stats_df.groupby("prefix").agg(
        files=("filename", "count"),
        median_fraction_abs_ge_0999=("fraction_abs_ge_0999", "median"),
        max_fraction_abs_ge_0999=("fraction_abs_ge_0999", "max"),
        median_fraction_abs_ge_099=("fraction_abs_ge_099", "median"),
        median_positive_limit=("fraction_at_positive_limit", "median"),
        median_negative_limit=("fraction_at_negative_limit", "median"),
    )
)

print("\nFiles with at least 0.1% of samples at |amplitude| >= 0.999:",
      int((stats_df["fraction_abs_ge_0999"] >= 0.001).sum()), "/", len(stats_df))

print("\n========== PAIRWISE VERSION COMPARISON ==========")
for numeric_id, versions in sorted(signals_by_id.items()):
    if not {"BP", "DP", "EP"}.issubset(versions):
        continue

    for a, b in [("BP", "DP"), ("BP", "EP"), ("DP", "EP")]:
        va, vb = versions[a], versions[b]
        xa, xb = va["signal"], vb["signal"]

        same_shape = len(xa) == len(xb) and va["sample_rate"] == vb["sample_rate"]
        corr = np.nan
        max_abs_diff = np.nan
        identical = False

        if same_shape:
            identical = bool(np.array_equal(xa, xb))
            max_abs_diff = float(np.max(np.abs(xa - xb)))
            if np.std(xa) > 0 and np.std(xb) > 0:
                corr = float(np.corrcoef(xa, xb)[0, 1])

        paired_stats.append({
            "numeric_id": numeric_id,
            "pair": f"{a}-{b}",
            "same_shape_and_rate": same_shape,
            "exactly_identical": identical,
            "correlation": corr,
            "max_abs_difference": max_abs_diff,
            "duration_difference_seconds": abs(va["duration"] - vb["duration"]),
        })

pairs_df = pd.DataFrame(paired_stats)

print("Total pair comparisons:", len(pairs_df))
print("Pairs with exactly identical signals:",
      int(pairs_df["exactly_identical"].sum()))
print("\n========== PAIRWISE SUMMARY ==========")
display(
    pairs_df.groupby("pair").agg(
        comparisons=("numeric_id", "count"),
        identical_pairs=("exactly_identical", "sum"),
        median_correlation=("correlation", "median"),
        min_correlation=("correlation", "min"),
        max_correlation=("correlation", "max"),
        median_max_abs_diff=("max_abs_difference", "median"),
        max_duration_difference=("duration_difference_seconds", "max"),
    )
)

print("\n========== EXAMPLE HIGH-SATURATION FILES ==========")
display(
    stats_df.sort_values("fraction_abs_ge_0999", ascending=False)
    [["filename", "prefix", "peak", "fraction_abs_ge_0999",
      "fraction_at_positive_limit", "fraction_at_negative_limit"]]
    .head(15)
)

print(
    "\nCaution: a sample near full scale is not necessarily clipped. "
    "A plateau at the amplitude limits and the source encoding are more "
    "informative than peak amplitude alone."
)


========== SAMPLE SATURATION SUMMARY ==========


,files,median_fraction_abs_ge_0999,max_fraction_abs_ge_0999,median_fraction_abs_ge_099,median_positive_limit,median_negative_limit
prefix,,,,,,
BP,112,0.000028,0.000100,0.000089,0.0,0.00001
DP,112,0.000017,0.000066,0.000044,0.0,0.00001
EP,112,0.000018,0.000060,0.000045,0.0,0.00001



Files with at least 0.1% of samples at |amplitude| >= 0.999: 0 / 336

========== PAIRWISE VERSION COMPARISON ==========
Total pair comparisons: 336
Pairs with exactly identical signals: 1

========== PAIRWISE SUMMARY ==========


,comparisons,identical_pairs,median_correlation,min_correlation,max_correlation,median_max_abs_diff,max_duration_difference
pair,,,,,,,
BP-DP,112,1,0.737842,0.401991,1.000000,0.795532,0.000
BP-EP,112,0,0.791449,0.460458,0.942632,0.827286,5.568
DP-EP,112,0,0.941386,0.761492,0.960469,0.353363,5.568



========== EXAMPLE HIGH-SATURATION FILES ==========


,filename,prefix,peak,fraction_abs_ge_0999,fraction_at_positive_limit,fraction_at_negative_limit
26,"BP22_BRON,Crep,P R U,24,M.wav",BP,1.000000,0.000100,0.000000,0.000100
10,"BP10_Asthma,E W,P R U,59,M.wav",BP,1.000000,0.000084,0.000000,0.000084
3,"BP103_N,N,P R U,81,F.wav",BP,0.999969,0.000078,0.000078,0.000000
51,"BP45_heart failure,Crep,P R L ,30,M.wav",BP,1.000000,0.000077,0.000000,0.000077
29,"BP25_copd,E W,P L L,76,M.wav",BP,0.999969,0.000066,0.000066,0.000000
141,"DP25_copd,E W,P L L,76,M.wav",DP,0.999969,0.000066,0.000066,0.000000
4,"BP104_Asthma,E W,P L U,45,F.wav",BP,1.000000,0.000064,0.000000,0.000064
63,"BP56_asthma,E W,P R M,56,F.wav",BP,1.000000,0.000063,0.000000,0.000063
35,"BP30_N,N,P R M,18,F.wav",BP,1.000000,0.000062,0.000000,0.000062
103,"BP92_N,N,P L L ,43,M.wav",BP,0.999969,0.000060,0.000060,0.000000



Caution: a sample near full scale is not necessarily clipped. A plateau at the amplitude limits and the source encoding are more informative than peak amplitude alone.


In [15]:

# Cell 14 — Identify subjects with duration differences and identical versions

print("========== LARGEST DURATION DIFFERENCES ==========")
duration_diff = (
    pairs_df.sort_values("duration_difference_seconds", ascending=False)
    .loc[
        :,
        ["numeric_id", "pair", "duration_difference_seconds",
         "correlation", "max_abs_difference", "exactly_identical"]
    ]
)
display(duration_diff.head(20))

print("\n========== COUNTS OF NONZERO DURATION DIFFERENCES ==========")
display(
    pairs_df.assign(
        duration_diff_nonzero=(
            pairs_df["duration_difference_seconds"] > 1e-6
        )
    )
    .groupby("pair")
    .agg(
        comparisons=("numeric_id", "count"),
        nonzero_duration_differences=("duration_diff_nonzero", "sum"),
        maximum_difference_seconds=("duration_difference_seconds", "max"),
    )
)

print("\n========== EXACTLY IDENTICAL PAIRS ==========")
display(
    pairs_df.loc[
        pairs_df["exactly_identical"],
        ["numeric_id", "pair", "correlation",
         "max_abs_difference", "duration_difference_seconds"]
    ]
)

print("\n========== ALL THREE VERSION DURATIONS FOR AFFECTED SUBJECTS ==========")
affected_ids = sorted(
    pairs_df.loc[
        pairs_df["duration_difference_seconds"] > 1e-6,
        "numeric_id"
    ].unique()
)

duration_rows = []
for numeric_id in affected_ids:
    versions = signals_by_id[numeric_id]
    row = {"numeric_id": numeric_id}
    for prefix in ["BP", "DP", "EP"]:
        if prefix in versions:
            row[f"{prefix}_duration_s"] = versions[prefix]["duration"]
            row[f"{prefix}_samples"] = len(versions[prefix]["signal"])
    duration_rows.append(row)

display(pd.DataFrame(duration_rows).head(30))

print(
    "\nThese are diagnostic findings only. Do not trim, resample, or overwrite "
    "any recordings until the version-generation procedure is understood."
)


========== LARGEST DURATION DIFFERENCES ==========


,numeric_id,pair,duration_difference_seconds,correlation,max_abs_difference,exactly_identical
227,76,DP-EP,5.568,NaN,NaN,False
226,76,BP-EP,5.568,NaN,NaN,False
298,100,BP-EP,1.408,NaN,NaN,False
299,100,DP-EP,1.408,NaN,NaN,False
0,1,BP-DP,0.000,0.742706,0.777374,False
220,74,BP-EP,0.000,0.780535,0.856110,False
225,76,BP-DP,0.000,0.746988,0.742249,False
224,75,DP-EP,0.000,0.945642,0.377960,False
223,75,BP-EP,0.000,0.766694,1.457947,False
222,75,BP-DP,0.000,0.720583,1.420807,False



========== COUNTS OF NONZERO DURATION DIFFERENCES ==========


,comparisons,nonzero_duration_differences,maximum_difference_seconds
pair,,,
BP-DP,112,0,0.000
BP-EP,112,2,5.568
DP-EP,112,2,5.568



========== EXACTLY IDENTICAL PAIRS ==========


,numeric_id,pair,correlation,max_abs_difference,duration_difference_seconds
24,9,BP-DP,1.0,0.0,0.0



========== ALL THREE VERSION DURATIONS FOR AFFECTED SUBJECTS ==========


,numeric_id,BP_duration_s,BP_samples,DP_duration_s,DP_samples,EP_duration_s,EP_samples
0,76,20.144,80576,20.144,80576,14.576,58304
1,100,13.552,54208,13.552,54208,14.960,59840



These are diagnostic findings only. Do not trim, resample, or overwrite any recordings until the version-generation procedure is understood.


In [16]:

# Cell 15 — Inspect exceptional subjects and their three recording versions

import numpy as np
import pandas as pd
import soundfile as sf
import re

subjects_to_inspect = [9, 76, 100]
rows = []

for numeric_id in subjects_to_inspect:
    print(f"\n========== SUBJECT {numeric_id} ==========")

    versions = {}
    for prefix in ["BP", "DP", "EP"]:
        matches = [
            p for p in fraiwan_root.rglob("*.wav")
            if re.match(rf"^{prefix}{numeric_id}_", p.name)
        ]

        if len(matches) != 1:
            print(f"{prefix}: expected one file, found {len(matches)}")
            continue

        path = matches[0]
        x, sr = sf.read(str(path), dtype="float32", always_2d=True)
        x = x[:, 0]

        versions[prefix] = x
        rows.append({
            "numeric_id": numeric_id,
            "prefix": prefix,
            "filename": path.name,
            "bytes": path.stat().st_size,
            "sample_rate": sr,
            "samples": len(x),
            "duration_s": len(x) / sr,
            "min_amplitude": float(np.min(x)),
            "max_amplitude": float(np.max(x)),
            "rms": float(np.sqrt(np.mean(x ** 2))),
            "std": float(np.std(x)),
        })

    if len(versions) == 3:
        print("Pairwise signal comparison:")
        for a, b in [("BP", "DP"), ("BP", "EP"), ("DP", "EP")]:
            xa, xb = versions[a], versions[b]
            same_length = len(xa) == len(xb)
            exactly_equal = same_length and np.array_equal(xa, xb)

            corr = np.nan
            if same_length and np.std(xa) > 0 and np.std(xb) > 0:
                corr = float(np.corrcoef(xa, xb)[0, 1])

            print({
                "pair": f"{a}-{b}",
                "same_length": same_length,
                "exactly_equal": exactly_equal,
                "correlation_if_same_length": corr,
            })

details = pd.DataFrame(rows)

print("\n========== FILE DETAILS ==========")
display(details)

print(
    "\nThis check is diagnostic only. Do not trim, resample, normalize, "
    "or replace any recordings based on these results."
)



========== SUBJECT 9 ==========
Pairwise signal comparison:
{'pair': 'BP-DP', 'same_length': True, 'exactly_equal': True, 'correlation_if_same_length': 1.0}
{'pair': 'BP-EP', 'same_length': True, 'exactly_equal': False, 'correlation_if_same_length': 0.7883699230379505}
{'pair': 'DP-EP', 'same_length': True, 'exactly_equal': False, 'correlation_if_same_length': 0.7883699230379505}

========== SUBJECT 76 ==========
Pairwise signal comparison:
{'pair': 'BP-DP', 'same_length': True, 'exactly_equal': False, 'correlation_if_same_length': 0.7469875096883877}
{'pair': 'BP-EP', 'same_length': False, 'exactly_equal': False, 'correlation_if_same_length': nan}
{'pair': 'DP-EP', 'same_length': False, 'exactly_equal': False, 'correlation_if_same_length': nan}

========== SUBJECT 100 ==========
Pairwise signal comparison:
{'pair': 'BP-DP', 'same_length': True, 'exactly_equal': False, 'correlation_if_same_length': 0.7430090304329077}
{'pair': 'BP-EP', 'same_length': False, 'exactly_equal': False, 'co

,numeric_id,prefix,filename,bytes,sample_rate,samples,duration_s,min_amplitude,max_amplitude,rms,std
0,9,BP,"BP9_Asthma,E W,P R L ,59,M.wav",122796,4000,61376,15.344,-0.926117,0.999969,0.087568,0.087555
1,9,DP,"DP9_Asthma,E W,P R L ,59,M.wav",122796,4000,61376,15.344,-0.926117,0.999969,0.087568,0.087555
2,9,EP,"EP9_Asthma,E W,P R L ,59,M.wav",122796,4000,61376,15.344,-0.835999,0.999969,0.075287,0.075276
3,76,BP,"BP76_N,N,A L U,31,M.wav",161196,4000,80576,20.144,-1.000000,0.958679,0.185447,0.185376
4,76,DP,"DP76_N,N,A L U,31,M.wav",161196,4000,80576,20.144,-1.000000,0.981995,0.176342,0.176318
5,76,EP,"EP76_N,N,A L U,31,M.wav",116652,4000,58304,14.576,-1.000000,0.806427,0.062650,0.062645
6,100,BP,"BP100_N,N,P R M,70,F.wav",108460,4000,54208,13.552,-0.878571,0.999969,0.228767,0.227783
7,100,DP,"DP100_N,N,P R M,70,F.wav",108460,4000,54208,13.552,-0.890747,0.999969,0.217410,0.217064
8,100,EP,"EP100_N,N,P R M,70,F.wav",119724,4000,59840,14.960,-0.928192,0.999969,0.070014,0.069574



This check is diagnostic only. Do not trim, resample, normalize, or replace any recordings based on these results.


In [17]:

# Cell 16 — Inventory Fraiwan documentation and archive contents

from pathlib import Path
import zipfile

candidate_roots = [
    PROJECT_ROOT / "data" / "raw" / "Fraiwan",
    PROJECT_ROOT / "data" / "raw" / "FRAIWAN",
]

for root in candidate_roots:
    print(f"\n========== ROOT: {root} ==========")
    print("Exists:", root.exists())

    if not root.exists():
        continue

    for path in sorted(root.iterdir()):
        if path.is_file():
            print(f"FILE  {path.name}  ({path.stat().st_size:,} bytes)")
        elif path.is_dir():
            print(f"DIR   {path.name}/")

    print("\nPotential documentation and metadata files:")
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue

        name = path.name.lower()
        if (
            path.suffix.lower() in {
                ".pdf", ".txt", ".md", ".csv", ".xlsx", ".xls",
                ".json", ".xml", ".yaml", ".yml"
            }
            or any(term in name for term in [
                "readme", "license", "metadata", "annotation",
                "description", "filter", "dataset"
            ])
        ):
            print(f"  {path.relative_to(root)}")

    print("\nZIP archive contents:")
    for archive in sorted(root.rglob("*.zip")):
        print(f"\nArchive: {archive.relative_to(root)}")
        try:
            with zipfile.ZipFile(archive) as zf:
                members = zf.infolist()
                print("Entries:", len(members))
                for item in members[:30]:
                    print(f"  {item.filename} ({item.file_size:,} bytes)")
                if len(members) > 30:
                    print(f"  ... {len(members) - 30} more entries")
        except Exception as exc:
            print("Could not inspect archive:", repr(exc))



========== ROOT: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/Fraiwan ==========
Exists: True
FILE  .DS_Store  (8,196 bytes)
DIR   Audio Files/
FILE  Data annotation (1).xlsx  (12,785 bytes)
FILE  Stethoscope Files.zip  (27,755,197 bytes)

Potential documentation and metadata files:
  Data annotation (1).xlsx

ZIP archive contents:

Archive: Stethoscope Files.zip
Entries: 11
  P101-P112.zsa (3,678,864 bytes)
  P1-P8.zsa (5,173,312 bytes)
  P21-P32.zsa (3,190,240 bytes)
  P33-P40.zsa (2,133,886 bytes)
  P41-P52.zsa (3,350,028 bytes)
  P53-P64.zsa (3,761,954 bytes)
  P65-P76.zsa (4,207,774 bytes)
  P77-P88.zsa (3,440,890 bytes)
  P89-P100.zsa (3,716,218 bytes)
  P9-P20.zsa (3,874,998 bytes)
  Readme.txt (290 bytes)

========== ROOT: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/FRAIWAN ==========
Exists: True
FILE  .DS_Store  (8,196 bytes)
DIR   Audio Files/
FILE  Data annotation (1).xlsx  (12,785 bytes)
FILE  Stethoscope Files.zip  (27,755,197 bytes)

Poten

In [18]:

# Cell 17 — Compare Fraiwan folder contents and read the stethoscope archive README

from pathlib import Path
import hashlib
import zipfile

roots = {
    "Fraiwan": PROJECT_ROOT / "data" / "raw" / "Fraiwan",
    "FRAIWAN": PROJECT_ROOT / "data" / "raw" / "FRAIWAN",
}

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

print("========== TOP-LEVEL FILE COMPARISON ==========")

for filename in [
    "Data annotation (1).xlsx",
    "Stethoscope Files.zip",
]:
    print(f"\n{filename}")
    paths = [roots[name] / filename for name in roots]

    for name, path in zip(roots, paths):
        print(
            name,
            "exists=", path.exists(),
            "bytes=", path.stat().st_size if path.exists() else None,
            "sha256=", sha256_file(path) if path.exists() else None,
        )

    if all(path.exists() for path in paths):
        print("Byte-identical:", sha256_file(paths[0]) == sha256_file(paths[1]))

print("\n========== AUDIO FILE INVENTORY COMPARISON ==========")

audio_maps = {}
for name, root in roots.items():
    audio_maps[name] = {
        p.relative_to(root / "Audio Files").as_posix(): p
        for p in (root / "Audio Files").rglob("*")
        if p.is_file()
    }
    print(f"{name}: {len(audio_maps[name])} files")

names_a = set(audio_maps["Fraiwan"])
names_b = set(audio_maps["FRAIWAN"])
print("Same relative filenames:", names_a == names_b)
print("Only in Fraiwan:", sorted(names_a - names_b)[:20])
print("Only in FRAIWAN:", sorted(names_b - names_a)[:20])

if names_a == names_b:
    mismatches = []
    for relpath in sorted(names_a):
        hash_a = sha256_file(audio_maps["Fraiwan"][relpath])
        hash_b = sha256_file(audio_maps["FRAIWAN"][relpath])
        if hash_a != hash_b:
            mismatches.append(relpath)

    print("Audio files with different SHA-256:", len(mismatches))
    print("Examples:", mismatches[:20])

print("\n========== ARCHIVE README ==========")

archive = roots["Fraiwan"] / "Stethoscope Files.zip"
with zipfile.ZipFile(archive) as zf:
    readme_candidates = [
        name for name in zf.namelist()
        if Path(name).name.lower() == "readme.txt"
    ]

    if not readme_candidates:
        print("No Readme.txt found in archive.")
    else:
        for readme_name in readme_candidates:
            print(f"\n--- {readme_name} ---")
            print(zf.read(readme_name).decode("utf-8", errors="replace"))

print(
    "\nNo archive contents were extracted. Hashes establish byte identity, "
    "not dataset provenance or the meaning of filename prefixes."
)


========== TOP-LEVEL FILE COMPARISON ==========

Data annotation (1).xlsx
Fraiwan exists= True bytes= 12785 sha256= a54f46a5fb6b2cf290ad7edc6c97ad1524be109dfe71054922c5693b918bf4cf
FRAIWAN exists= True bytes= 12785 sha256= a54f46a5fb6b2cf290ad7edc6c97ad1524be109dfe71054922c5693b918bf4cf
Byte-identical: True

Stethoscope Files.zip
Fraiwan exists= True bytes= 27755197 sha256= 9cd1449eda34ef5615800fecc344d06dddf5bafd1bb65cc97b3ae58b252c65b3
FRAIWAN exists= True bytes= 27755197 sha256= 9cd1449eda34ef5615800fecc344d06dddf5bafd1bb65cc97b3ae58b252c65b3
Byte-identical: True

========== AUDIO FILE INVENTORY COMPARISON ==========
Fraiwan: 336 files
FRAIWAN: 336 files
Same relative filenames: True
Only in Fraiwan: []
Only in FRAIWAN: []
Audio files with different SHA-256: 0
Examples: []

========== ARCHIVE README ==========

--- Readme.txt ---
1. First four recordings from the file P1-P8.zsa were used for testing the stethoscope and should be ignored.
2. The files are named by the included patien

In [19]:

# Cell 18 — Inspect original archive structure and patient-range coverage

from pathlib import Path
import zipfile
import re

archive_path = (
    PROJECT_ROOT / "data" / "raw" / "Fraiwan" / "Stethoscope Files.zip"
)

with zipfile.ZipFile(archive_path) as zf:
    members = zf.infolist()

    print("========== ARCHIVE SUMMARY ==========")
    print("Archive:", archive_path.name)
    print("Total entries:", len(members))

    print("\n========== ENTRY DETAILS ==========")
    for item in members:
        print({
            "name": item.filename,
            "uncompressed_bytes": item.file_size,
            "compressed_bytes": item.compress_size,
            "is_directory": item.is_dir(),
        })

    print("\n========== PATIENT RANGE COVERAGE ==========")
    ranges = []

    for item in members:
        name = Path(item.filename).name
        match = re.fullmatch(r"P(\d+)-P(\d+)\.zsa", name, flags=re.IGNORECASE)

        if match:
            start, end = map(int, match.groups())
            ranges.append({
                "archive": name,
                "start_patient": start,
                "end_patient": end,
                "nominal_patient_count": end - start + 1,
                "uncompressed_bytes": item.file_size,
            })

    ranges_df = sorted(ranges, key=lambda x: x["start_patient"])
    for row in ranges_df:
        print(row)

    covered = set()
    for row in ranges_df:
        covered.update(range(row["start_patient"], row["end_patient"] + 1))

    print("\nUnique patient numbers covered by filenames:", len(covered))
    print("Minimum patient number:", min(covered) if covered else None)
    print("Maximum patient number:", max(covered) if covered else None)

    print(
        "\nThe range names describe the archive organization, not proof that "
        "every entry is a valid patient recording. The README's four-recording "
        "exclusion still needs to be considered when interpreting P1-P8.zsa."
    )


========== ARCHIVE SUMMARY ==========
Archive: Stethoscope Files.zip
Total entries: 11

========== ENTRY DETAILS ==========
{'name': 'P101-P112.zsa', 'uncompressed_bytes': 3678864, 'compressed_bytes': 2795231, 'is_directory': False}
{'name': 'P1-P8.zsa', 'uncompressed_bytes': 5173312, 'compressed_bytes': 3930755, 'is_directory': False}
{'name': 'P21-P32.zsa', 'uncompressed_bytes': 3190240, 'compressed_bytes': 2423869, 'is_directory': False}
{'name': 'P33-P40.zsa', 'uncompressed_bytes': 2133886, 'compressed_bytes': 1621320, 'is_directory': False}
{'name': 'P41-P52.zsa', 'uncompressed_bytes': 3350028, 'compressed_bytes': 2545372, 'is_directory': False}
{'name': 'P53-P64.zsa', 'uncompressed_bytes': 3761954, 'compressed_bytes': 2858247, 'is_directory': False}
{'name': 'P65-P76.zsa', 'uncompressed_bytes': 4207774, 'compressed_bytes': 3197067, 'is_directory': False}
{'name': 'P77-P88.zsa', 'uncompressed_bytes': 3440890, 'compressed_bytes': 2614329, 'is_directory': False}
{'name': 'P89-P100.z